# Harlequin category romance: corpus and translation analysis

This notebook analyses the tables produced by the pipeline in this repository (see README.md):

| File | Content |
|---|---|
| `harlequin_lines.csv` (+ `english_additions.csv`) | English books of the selected Harlequin/Silhouette lines (+ originals found via Open Library) |
| `harlequin_lines_with_translations.csv` | the same books joined to their French / German / Polish editions |
| `unmatched_all.csv` | target-language records that could not be linked to an English book |

**Sections**
1. Setup and loading
2. The original corpus (timeline, length, ratings, authors)
3. Translation coverage (how much, when, how fast, by whom, by line)
4. Quality check of the matching (what may be missing)
5. Shortlist for the alignment study (books with parallel translations)
6. Exports

All figures are saved to `figures/` (PNG + SVG), all derived tables to `output/`.

## 1. Setup

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# --- file locations (change if your files live elsewhere) -----------------
DATA_DIR    = Path(".")          # folder with the pipeline outputs
# English book tables: one file, or several (e.g. the five lines + books found via Open Library)
BOOKS_CSV   = [DATA_DIR / "harlequin_lines.csv", DATA_DIR / "english_additions.csv"]
MERGED_CSV  = DATA_DIR / "harlequin_lines_with_translations.csv"
UNMATCH_CSV = DATA_DIR / "unmatched_all.csv"
FIG_DIR, OUT_DIR = Path("figures"), Path("output")
FIG_DIR.mkdir(exist_ok=True); OUT_DIR.mkdir(exist_ok=True)

# --- chart style ------------------------------------------------------------
# Colour = language identity, fixed order, never by rank. The three hues are
# validated for colour-blind separation (all pairs). Polish (aqua) is low-contrast
# on white, so every chart names the languages in a legend or direct label.
LANG_ORDER  = ["French", "German", "Polish"]
LANG_COLORS = {"French": "#2a78d6", "German": "#eb6834", "Polish": "#1baf7a"}
NEUTRAL     = "#2a78d6"          # single-series charts use slot 1
MUTED       = "#8a8984"
INK, INK2   = "#0b0b0b", "#52514e"

plt.rcParams.update({
    "figure.figsize": (10, 4.2), "figure.dpi": 110, "savefig.dpi": 200,
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#d6d5d0", "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.titlesize": 13, "axes.titleweight": "semibold", "axes.titlelocation": "left",
    "axes.labelsize": 10, "xtick.color": INK2, "ytick.color": INK2,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#ebeae6", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "lines.linewidth": 2, "legend.frameon": False, "legend.fontsize": 9,
    "font.family": "sans-serif",
})

def save(fig, name):
    fig.tight_layout()
    for ext in ("png", "svg"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", bbox_inches="tight")
    plt.show()

def subtitle(ax, text):
    # move the title up and put the subtitle in the gap, so they never overlap
    ax.set_title(ax.get_title(loc="left"), loc="left", pad=24)
    ax.text(0, 1.015, text, transform=ax.transAxes, fontsize=9.5, color=INK2, va="bottom")

## 2. Load and clean

In [ ]:
paths  = BOOKS_CSV if isinstance(BOOKS_CSV, (list, tuple)) else [BOOKS_CSV]
books  = pd.concat([pd.read_csv(p, encoding="utf-8-sig") for p in paths if Path(p).exists()], ignore_index=True)
merged = pd.read_csv(MERGED_CSV, encoding="utf-8-sig", low_memory=False)
# book ids are numbers (FictionDB) or text (Open Library, e.g. OL999W): compare them as text
books["book_id"]  = books["book_id"].astype(str)
merged["book_id"] = merged["book_id"].astype(str)
books = books.drop_duplicates("book_id")
if "line" not in books:
    books["line"] = "Harlequin American Romance"
books["line"] = books["line"].fillna("Other Harlequin (via Open Library)")
unm    = pd.read_csv(UNMATCH_CSV, encoding="utf-8-sig")

def parse_month_year(s):
    s = str(s).strip()
    for fmt in ("%b %Y", "%B %Y", "%Y"):
        try:
            return pd.to_datetime(s, format=fmt)
        except (ValueError, TypeError):
            pass
    return pd.NaT

# columns that only exist when the scraper ran with --details (rating is partly on list pages)
for col in ("rating", "pages"):
    books[col] = pd.to_numeric(books[col], errors="coerce") if col in books else np.nan
HAS_PAGES, HAS_RATING = books["pages"].notna().any(), books["rating"].notna().any()
if not (HAS_PAGES and HAS_RATING):
    print("Note: no page counts / few ratings (scraper ran without --details); those charts are skipped.")

books["pub_dt"]        = books["pub_date"].map(parse_month_year)
books["orig_year"]     = books["pub_dt"].dt.year.astype("Int64")
books["series_number"] = pd.to_numeric(books["series_number"], errors="coerce").astype("Int64")

# unit flags (added earlier; recomputed here if missing)
if "is_companion" not in books:
    books["is_companion"] = books["series_number"].isna()
if "is_anthology" not in books:
    books["is_anthology"] = books["series_number"].duplicated(keep=False) & books["series_number"].notna()

# The corpus for the study: numbered series titles plus books found via translations
# (anthology stories count individually; unnumbered companion titles are left out)
books["is_companion"] = books["is_companion"].astype(str).str.lower().eq("true")
corpus = books[~books["is_companion"]].copy()

# --- translations: one row per translated edition -------------------------------
tr = merged[merged["tr_translation_id"].notna()].copy()
# Translation columns are prefixed tr_. Book columns with the same name after stripping the
# prefix (e.g. the book's own publisher / pub_date) are renamed book_* so nothing clashes.
tr_names = {c.removeprefix("tr_") for c in tr.columns if c.startswith("tr_")}
tr = tr.rename(columns={c: f"book_{c}" for c in tr.columns if not c.startswith("tr_") and c in tr_names})
tr.columns = [c.removeprefix("tr_") for c in tr.columns]
for col in ("edition_type", "source", "works_in_edition", "first_edition", "first_edition_year", "original_series"):   # columns added in later script versions
    if col not in tr:
        tr[col] = np.nan
tr = tr.merge(books[["book_id", "orig_year", "rating", "pages"]], on="book_id", how="left",
              suffixes=("", "_book"))
tr["pub_year"] = pd.to_numeric(tr["pub_year"], errors="coerce").astype("Int64")
# translation year = first edition in the language when known (Cora reissues state it), else this edition's year
fe = pd.to_numeric(tr.get("first_edition_year"), errors="coerce") if "first_edition_year" in tr else np.nan
tr["translation_year"] = pd.Series(fe, index=tr.index).fillna(tr["pub_year"]).astype("Int64")
tr["lag_years"] = (tr["translation_year"] - tr["orig_year"]).astype("Int64")

def publisher_family(p):
    p = str(p).lower()
    if "harlequin" in p:            return "Harlequin"
    if "cora" in p:                 return "Cora"
    if "silhouette" in p or "awhe" in p: return "Silhouette / AWHE"
    if "weltbild" in p:             return "Weltbild"
    if p in ("nan", ""):            return "unknown"
    return str(p).strip().title()
tr["publisher_family"] = tr["publisher"].map(publisher_family)
tr["has_translator"]   = tr["translators"].notna() & (tr["translators"].astype(str).str.strip() != "")

print(f"Books: {len(books):,}  (numbered corpus: {len(corpus):,}, companions: {books['is_companion'].sum()}, "
      f"anthology stories: {books['is_anthology'].sum()})")
print(f"Translated editions: {len(tr):,} of {tr['book_id'].nunique():,} distinct books")
print(f"Unmatched target-language records: {len(unm):,}")

## 3. The original corpus
### 3.1 Publication timeline
How many series titles appeared per year? Anthology stories are counted individually.

In [ ]:
per_year = corpus.groupby("orig_year").size()
fig, ax = plt.subplots()
ax.bar(per_year.index, per_year.values, width=0.8, color=NEUTRAL)
ax.set_title("Titles per year" + (f" ({corpus.line.nunique()} lines)" if "line" in corpus else ""))
subtitle(ax, f"{len(corpus):,} numbered titles, {per_year.index.min()}–{per_year.index.max()}")
ax.set_xlabel("Year of original publication"); ax.set_ylabel("Titles")
save(fig, "01_titles_per_year")
per_year.describe().round(1).to_frame("titles per year").T

### 3.2 Book length over time
Page counts are from the first listed edition; the band shows the middle 50% of books per year. Page counts in the later years are often identical because FictionDB lists a standard edition size.

In [ ]:
if not HAS_PAGES:
    print('Skipped: this column is empty (run the scraper with --details to fill it).')
else:
    pg = corpus.dropna(subset=["pages", "orig_year"]).groupby("orig_year")["pages"]
    q = pd.DataFrame({"median": pg.median(), "q1": pg.quantile(.25), "q3": pg.quantile(.75), "n": pg.size()})
    fig, ax = plt.subplots()
    ax.fill_between(q.index, q.q1, q.q3, color=NEUTRAL, alpha=.18, linewidth=0, label="Middle 50%")
    ax.plot(q.index, q["median"], color=NEUTRAL, label="Median")
    ax.set_title("Page count per book over time")
    subtitle(ax, "Median and interquartile range per year of original publication")
    ax.set_xlabel("Year"); ax.set_ylabel("Pages"); ax.legend(loc="lower right")
    save(fig, "02_pages_over_time")

### 3.3 Reader ratings
FictionDB member ratings (1–5). Books without ratings are excluded.

In [ ]:
if not HAS_RATING:
    print('Skipped: this column is empty (run the scraper with --details to fill it).')
else:
    r = corpus["rating"].dropna()
    fig, ax = plt.subplots(figsize=(7, 3.6))
    counts = r.round(1).value_counts().sort_index()
    ax.bar(counts.index, counts.values, width=0.35, color=NEUTRAL)
    ax.axvline(r.median(), color=INK2, linestyle="--", linewidth=1)
    ax.text(r.median(), ax.get_ylim()[1]*0.95, f"  median {r.median():.1f}", color=INK2, fontsize=9, va="top")
    ax.set_title("Distribution of ratings")
    subtitle(ax, f"{len(r):,} rated books · {corpus['rating'].isna().sum()} unrated")
    ax.set_xlabel("Rating"); ax.set_ylabel("Books")
    save(fig, "03_ratings")

### 3.4 Authors
The series is written by many authors, but a few wrote a large share.

In [ ]:
auth = corpus["authors"].str.split(r";\s*").explode().value_counts()
top = auth.head(15)[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top.index, top.values, color=NEUTRAL, height=0.7)
for y, v in enumerate(top.values):
    ax.text(v + 1, y, str(v), va="center", fontsize=8.5, color=INK2)
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
ax.set_title("Most prolific authors in the series")
share = auth.head(15).sum() / auth.sum()
subtitle(ax, f"{len(auth)} authors in total · the top 15 wrote {share:.0%} of titles")
ax.set_xlabel("Titles")
save(fig, "04_top_authors")

# concentration: how many authors account for half of all titles?
cum = auth.cumsum() / auth.sum()
print(f"{(cum < .5).sum() + 1} authors wrote half of all titles; "
      f"{(auth == 1).sum()} authors wrote only one.")

## 4. Translation coverage
### 4.1 Summary per language

In [ ]:
n_corpus = len(corpus)
summary = (tr.groupby("language")
             .agg(editions=("translation_id", "size"),
                  books=("book_id", "nunique"),
                  first_year=("translation_year", "min"),
                  last_year=("pub_year", "max"),
                  median_lag_years=("lag_years", "median"),
                  translator_known=("has_translator", "mean"))
             .reindex(LANG_ORDER))
summary["share_of_corpus"] = summary["books"] / n_corpus
summary["books_with_several_editions"] = (tr.groupby(["language", "book_id"]).size() > 1).groupby("language").sum()
summary.style.format({"share_of_corpus": "{:.1%}", "translator_known": "{:.0%}", "median_lag_years": "{:.0f}"})

### 4.2 Which originals were translated?
Share of each year's titles with at least one edition in the language. Gaps can mean “not translated” **or** “not catalogued with an original title”, see section 5.

In [ ]:
translated = tr.groupby(["language", "book_id"]).size().reset_index()[["language", "book_id"]]
base = corpus[["book_id", "orig_year"]].dropna()
fig, ax = plt.subplots()
for lang in LANG_ORDER:
    ids = set(translated.loc[translated.language == lang, "book_id"])
    share = base.assign(t=base.book_id.isin(ids)).groupby("orig_year")["t"].mean()
    ax.plot(share.index, share.values * 100, color=LANG_COLORS[lang], label=lang,
            marker="o", markersize=3.5)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
ax.set_title("Share of each year's titles with a known translation")
subtitle(ax, "By year of the original English publication")
ax.set_xlabel("Year of original publication"); ax.set_ylabel("Titles translated")
ax.legend(loc="upper right")
save(fig, "05_translated_share_by_year")

### 4.3 When did translations appear, and how fast?
Left: translated editions per year. Right: years between the original and the translation (each dot is one edition). Where a publisher page names the first edition in that language (Cora reissues), that year is used instead of the reissue year.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.6, 1]})

years = range(int(tr.pub_year.min()), int(tr.pub_year.max()) + 1)
width = 0.28
for i, lang in enumerate(LANG_ORDER):
    c = tr[tr.language == lang].groupby("pub_year").size().reindex(years, fill_value=0)
    ax1.bar(np.array(list(years)) + (i - 1) * width, c.values, width=width - 0.02,
            color=LANG_COLORS[lang], label=lang)
ax1.set_title("Translated editions per year")
ax1.set_xlabel("Year of translated edition"); ax1.set_ylabel("Editions"); ax1.legend()

rng = np.random.default_rng(0)
for i, lang in enumerate(LANG_ORDER):
    lag = tr.loc[tr.language == lang, "lag_years"].dropna().astype(float)
    ax2.scatter(lag, i + rng.uniform(-.18, .18, len(lag)), s=22, color=LANG_COLORS[lang],
                edgecolor="#fcfcfb", linewidth=0.8, label=lang)
    if len(lag):
        ax2.plot([lag.median()] * 2, [i - .3, i + .3], color=INK, linewidth=2)
        ax2.text(lag.median(), i + .36, f"median {lag.median():.0f} y · n={len(lag)}",
                 ha="center", fontsize=8.5, color=INK2)
ax2.set_yticks(range(len(LANG_ORDER)), LANG_ORDER)
ax2.set_ylim(-.6, len(LANG_ORDER) - .3)
ax2.grid(axis="y", visible=False); ax2.grid(axis="x", visible=True)
ax2.set_title("Translation lag"); ax2.set_xlabel("Years after the original")
save(fig, "06_translation_timing")

### 4.4 Who published the translations?

In [ ]:
pub = (tr.groupby(["publisher_family", "language"]).size().unstack(fill_value=0)
         .reindex(columns=LANG_ORDER, fill_value=0))
pub = pub.loc[pub.sum(axis=1).sort_values().index]
fig, ax = plt.subplots(figsize=(8, max(2.5, 0.45 * len(pub) + 1)))
left = np.zeros(len(pub))
for lang in LANG_ORDER:
    v = pub[lang].values
    ax.barh(pub.index, v, left=left, color=LANG_COLORS[lang], label=lang, height=0.65,
            edgecolor="#fcfcfb", linewidth=2)
    left += v
for y, total in enumerate(pub.sum(axis=1)):
    ax.text(total + 1, y, str(total), va="center", fontsize=8.5, color=INK2)
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
ax.set_xlim(0, pub.sum(axis=1).max() * 1.08)
ax.set_title("Publishers of the translated editions"); ax.set_xlabel("Editions")
ax.legend(loc="lower right")
save(fig, "07_publishers")

print("Most frequent target-language series:")
tr["series"].fillna("").str.replace(r"\s*\d+.*$", "", regex=True).replace("", np.nan) \
    .value_counts().head(10)

### 4.5 Which authors were translated most?

In [ ]:
ta = (tr.groupby(["author", "language"])["book_id"].nunique().unstack(fill_value=0)
        .reindex(columns=LANG_ORDER, fill_value=0))
ta["total"] = ta.sum(axis=1)
ta = ta.sort_values("total").tail(15)
fig, ax = plt.subplots(figsize=(8, 5))
left = np.zeros(len(ta))
for lang in LANG_ORDER:
    ax.barh(ta.index, ta[lang], left=left, color=LANG_COLORS[lang], label=lang, height=0.7,
            edgecolor="#fcfcfb", linewidth=2)
    left += ta[lang].values
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
for y, total in enumerate(ta["total"]):
    ax.text(total + 0.2, y, str(total), va="center", fontsize=8.5, color=INK2)
ax.set_xlim(0, ta["total"].max() * 1.1)
ax.set_title("Authors with the most translated series titles")
subtitle(ax, "Distinct series titles with at least one edition, per language")
ax.set_xlabel("Translated titles"); ax.legend(loc="lower right")
save(fig, "08_translated_authors")

# translation rate relative to how much each author wrote
rate = (corpus["authors"].str.split(r";\s*").explode().value_counts().rename("titles").to_frame()
          .join(tr.groupby("author")["book_id"].nunique().rename("translated_titles"))
          .fillna(0))
rate["share_translated"] = rate["translated_titles"] / rate["titles"]
rate.query("titles >= 10").sort_values("share_translated", ascending=False).head(15) \
    .style.format({"share_translated": "{:.0%}", "translated_titles": "{:.0f}"})

### 4.6 Are translated books different?
Comparing books with and without a known translation on rating and length. This is descriptive only: translation choices also depend on era and on catalogue coverage.

In [ ]:
corpus["translated_any"] = corpus["book_id"].isin(tr["book_id"])
cols = [(col, label) for col, label, ok in [("rating", "Rating", HAS_RATING), ("pages", "Pages", HAS_PAGES)] if ok]
if not cols:
    print("Skipped: no ratings or page counts (run the scraper with --details).")
fig, axes = plt.subplots(1, max(len(cols), 1), figsize=(5.5 * max(len(cols), 1), 3.8), squeeze=False)
axes = axes[0]
for ax, (col, label) in zip(axes, cols):
    groups = [corpus.loc[~corpus.translated_any, col].dropna(), corpus.loc[corpus.translated_any, col].dropna()]
    bp = ax.boxplot(groups, vert=False, widths=0.5, patch_artist=True, showfliers=False,
                    medianprops=dict(color=INK, linewidth=2))
    for patch, c in zip(bp["boxes"], [MUTED, NEUTRAL]):
        patch.set_facecolor(c); patch.set_alpha(.35); patch.set_edgecolor(c)
    ax.set_yticks([1, 2], [f"not translated\n(n={len(groups[0])})", f"translated\n(n={len(groups[1])})"])
    ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
    ax.set_title(label)
if cols:
    save(fig, "09_translated_vs_not")
else:
    plt.close(fig)

try:
    from scipy.stats import mannwhitneyu
    for col, _ in cols:
        a = corpus.loc[corpus.translated_any, col].dropna(); b = corpus.loc[~corpus.translated_any, col].dropna()
        print(f"{col:7s} median translated {a.median():.1f} vs not {b.median():.1f}  "
              f"(Mann-Whitney p = {mannwhitneyu(a, b).pvalue:.3g})")
except ImportError:
    print("Install scipy for the significance test.")

### 4.7 Parallel coverage across languages
For an alignment study, books available in **several** languages are the most valuable.

In [ ]:
langs_per_book = tr.groupby("book_id")["language"].agg(lambda s: " + ".join(l for l in LANG_ORDER if l in set(s)))
combo = langs_per_book.value_counts().sort_values()
fig, ax = plt.subplots(figsize=(8, max(2.5, 0.5 * len(combo) + 1)))
ax.barh(combo.index, combo.values, color=NEUTRAL, height=0.6)
for y, v in enumerate(combo.values):
    ax.text(v + 0.5, y, str(v), va="center", fontsize=8.5, color=INK2)
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
ax.set_title("Books by combination of available translations")
subtitle(ax, f"{len(langs_per_book)} books with at least one translation")
ax.set_xlabel("Books")
save(fig, "10_language_combinations")

### 4.8 Translation coverage by line
Share of each line's books with at least one known edition in the language.

In [ ]:
lines_order = corpus["line"].value_counts().index.tolist()
cov = pd.DataFrame({lang: corpus.groupby("line")["book_id"].apply(
          lambda ids, l=lang: ids.isin(tr.loc[tr.language == l, "book_id"]).mean()) for lang in LANG_ORDER}).reindex(lines_order)
fig, ax = plt.subplots(figsize=(9, max(3, 0.8 * len(cov) + 1)))
h = 0.26
y = np.arange(len(cov))
for i, lang in enumerate(LANG_ORDER):
    ax.barh(y + (1 - i) * h, cov[lang] * 100, height=h - 0.03, color=LANG_COLORS[lang], label=lang)
    for yy, v in zip(y + (1 - i) * h, cov[lang] * 100):
        ax.text(v + 0.4, yy, f"{v:.0f}%", va="center", fontsize=8, color=INK2)
ax.set_yticks(y, [f"{l}\n(n={int((corpus.line == l).sum()):,})" for l in cov.index])
ax.invert_yaxis()
ax.xaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
ax.set_title("Share of books with a known translation, by line")
ax.set_xlabel("Books with at least one edition"); ax.legend(loc="lower right")
save(fig, "13_coverage_by_line")
(cov * 100).round(1)

## 5. Quality check of the matching
The automatic matching only finds translations whose library record names the English original.
This section shows what may be missing and where manual review pays off.

In [ ]:
unm["language"] = unm["language_letter"].map({"F": "French", "G": "German", "P": "Polish"})
reasons = unm.groupby(["language", "reason"]).size().unstack(fill_value=0).reindex(LANG_ORDER)
reason_colors = {"no original title in record": MUTED, "title did not match": NEUTRAL}
fig, ax = plt.subplots(figsize=(8, 2.8))
left = np.zeros(len(reasons))
for reason in reasons.columns:
    ax.barh(reasons.index, reasons[reason], left=left, color=reason_colors.get(reason, INK2),
            label=reason, height=0.6, edgecolor="#fcfcfb", linewidth=2)
    left += reasons[reason].values
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
ax.set_title("Why library records were not matched")
subtitle(ax, "Target-language records by the series authors that could not be linked to a series title")
ax.set_xlabel("Records")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.32), ncol=2)
save(fig, "11_unmatched_reasons")
reasons

**Wrong-person hits.** The French catalogue search matches name words in any order, so records by *other* people
(e.g. “Morgan Matson” for the author “Alice Morgan”) end up in the unmatched list. Records whose statement of
responsibility does not contain the author's surname are flagged here; they can be ignored.

In [ ]:
def surname_in_resp(row):
    surname = str(row["author"]).split()[-1].lower()
    return surname in str(row["responsibility"]).lower() or pd.isna(row["responsibility"])

unm["author_confirmed"] = unm.apply(surname_in_resp, axis=1)
unm.groupby("language")["author_confirmed"].agg(records="size", author_in_record="mean") \
   .reindex(LANG_ORDER).style.format({"author_in_record": "{:.0%}"})

**Near misses.** Records whose recorded original title is similar, but below the match threshold (0.88). These are the best candidates for manual checking.

In [ ]:
THRESHOLD = 0.88
real = unm[unm["author_confirmed"]]
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.hist(real.loc[real.reason == "title did not match", "closest_score"], bins=np.arange(0, 1.01, .02),
        color=NEUTRAL, edgecolor="#fcfcfb", linewidth=1)
ax.axvline(THRESHOLD, color=INK2, linestyle="--", linewidth=1)
ax.text(THRESHOLD, ax.get_ylim()[1] * .95, "  match threshold", color=INK2, fontsize=9, va="top")
ax.set_title("Similarity of unmatched original titles to the closest series title")
subtitle(ax, "Records by the right author that name an original title")
ax.set_xlabel("Title similarity (0–1)"); ax.set_ylabel("Records")
save(fig, "12_near_miss_scores")

near = (real[(real.reason == "title did not match") & (real.closest_score >= 0.7)]
        .sort_values("closest_score", ascending=False)
        [["author", "language", "translated_title", "original_title_in_record",
          "closest_series_title", "closest_score", "pub_year", "publisher", "source_record_id"]])
near.to_csv(OUT_DIR / "near_misses_to_review.csv", index=False, encoding="utf-8-sig")
print(f"{len(near)} near misses saved to output/near_misses_to_review.csv")
near.head(15)

**Records without an original title.** Right author, right language, but the library did not record the English title. Matching these needs the review loop (`candidate_matches.csv`) or another source.

In [ ]:
no_orig = unm[(unm.reason == "no original title in record") & unm.author_confirmed]
print(no_orig.groupby("language").size().reindex(LANG_ORDER).rename("records").to_string())
print("\nMost common publishers among them:")
no_orig["publisher"].fillna("unknown").value_counts().head(10)

## 6. Shortlist for the alignment study
Books ranked by how many languages they are available in, then by whether translators are known
(useful for studying translator effects) and by rating. One row per book, with every edition's ID listed.

In [ ]:
def join_unique(s):
    return "; ".join(dict.fromkeys(str(x) for x in s if pd.notna(x) and str(x).strip()))

short = (tr.groupby("book_id")
           .agg(title=("title", "first"), author=("author", "first"), orig_year=("orig_year", "first"),
                series_number=("series_number", "first"), rating=("rating", "first"),
                n_languages=("language", "nunique"),
                languages=("language", lambda s: " + ".join(l for l in LANG_ORDER if l in set(s))),
                n_editions=("translation_id", "size"),
                translation_ids=("translation_id", join_unique),
                translated_titles=("translated_title", join_unique),
                translators=("translators", join_unique),
                publishers=("publisher", join_unique),
                translation_years=("pub_year", join_unique),
                edition_types=("edition_type", join_unique),
                sources=("source", join_unique))
           .reset_index())
short["translator_known"] = short["translators"] != ""
short = short.sort_values(["n_languages", "translator_known", "n_editions", "rating"],
                          ascending=[False, False, False, False])
short.to_csv(OUT_DIR / "alignment_candidates.csv", index=False, encoding="utf-8-sig")
print(f"{len(short)} books saved to output/alignment_candidates.csv; "
      f"{(short.n_languages >= 2).sum()} are available in 2+ languages.")
short.head(20)

## 7. Exports

In [ ]:
tr.to_csv(OUT_DIR / "translations_clean.csv", index=False, encoding="utf-8-sig")
corpus.to_csv(OUT_DIR / "corpus_with_flags.csv", index=False, encoding="utf-8-sig")
summary.to_csv(OUT_DIR / "summary_per_language.csv", encoding="utf-8-sig")
print("Saved:", *sorted(p.name for p in OUT_DIR.iterdir()), sep="\n  ")
print("Figures:", len(list(FIG_DIR.glob('*.png'))), "PNG in", FIG_DIR)

## Next steps
1. **Review** `output/near_misses_to_review.csv` and `candidate_matches.csv`. Confirm correct pairs, then re-run `find_translations.py` so they enter the tables.
2. **Extend the search** for German with a second source for Cora magazine novels (the DNB often catalogues them without the original title).
3. **Choose the study sample** from `output/alignment_candidates.csv`, for example books with French + German editions and known translators.
4. **Get the texts** (e-books or scans) for the sample, then segment and align them (e.g. sentence alignment with *bertalign* or *hunalign*).